# Multi-classification Task

In [1]:
import numpy
import pandas # or use `polars`
from barbor import barbor

## Data Cleaning and Processing

### Load Dataset from CSV File by IO Process

In [ ]:
# https://github.com/linjing-lab/easy-pytorch/blob/main/released_box/data/bitcoin_heist_data.csv
df = pandas.read_csv('./bitcoin_heist_data.csv')
df.head()

,address,year,day,length,weight,count,looped,neighbors,income,label
0,111K8kZAEnJg245r2cM6y9zgJGHZtJPy6,2017,11,18,0.008333,1,0,2,100050000.0,princetonCerber
1,1123pJv8jzeFQaCV4w644pzQJzVWay2zcA,2016,132,44,0.000244,1,0,1,100000000.0,princetonLocky
2,112536im7hy6wtKbpH1qYDWtTyMRAcA2p7,2016,246,0,1.000000,1,0,2,200000000.0,princetonCerber
3,1126eDRw2wqSkWosjTCre8cjjQW8sSeWH7,2016,322,72,0.003906,1,0,2,71200000.0,princetonCerber
4,1129TSjKtx65E35GiUo4AYVeyo48twbrGX,2016,238,144,0.072848,456,0,1,200000000.0,princetonLocky


### Convert Any Format of Data to Numpy 

In [5]:
df = df.to_numpy()
labels = df[:,-1]

### Processing Data and Obtain Dataset Information

In [6]:
features = df[:,1:-1].astype(numpy.float64)
print(features.shape)

(1048575, 8)


In [7]:
print(features.shape[0], features.shape[1])
print(labels.shape, len(numpy.unique(labels)))

1048575 8
(1048575,) 29


## Machine Learning Process

### Load Perming and Config Hyperparameters

In [8]:
import perming
main = perming.Box(8, 29, (60,), batch_size=256, activation='relu', inplace_on=True, solver='adam', learning_rate_init=0.01)
main.solver = barbor(
        main.model.parameters(),
        lr=main.lr,
        method='alternating',
        gamma=1e-8,
        min_step=1e-8,
        max_step=1e3,
        adaptive_restart=True,
        restart_condition='both',
        restart_tol=0.9,
        momentum=0.0,
        nesterov=False
    )
main.print_config()

MLP(
  (mlp): Sequential(
    (Linear0): Linear(in_features=8, out_features=60, bias=True)
    (Activation0): ReLU(inplace=True)
    (Linear1): Linear(in_features=60, out_features=29, bias=True)
  )
)


OrderedDict([('torch -v', '1.7.1+cu101'),
             ('criterion', CrossEntropyLoss()),
             ('batch_size', 256),
             ('solver',
              barbor (
              Parameter Group 0
                  adaptive_restart: True
                  dampening: 0.0
                  gamma: 1e-08
                  lr: 0.01
                  max_step: 1000.0
                  method: alternating
                  min_step: 1e-08
                  momentum: 0.0
                  nesterov: False
                  restart_condition: both
                  restart_tol: 0.9
                  safe_guard: 1e-08
              )),
             ('lr_scheduler', None),
             ('device', device(type='cuda'))])

### Dataloader from Numpy with Multi-threaded

In [9]:
main.data_loader(features, labels, random_seed=0)

### Training Stage and Accelerated Validation

In [11]:
main.train_val(num_epochs=1, interval=100, backend='threading', early_stop=True) # set n_jobs > 1 within number of processes

Epoch [1/1], Step [100/3277], Training Loss: 0.3057, Validation Loss: 0.0344
Epoch [1/1], Step [200/3277], Training Loss: 0.2265, Validation Loss: 0.0729
Epoch [1/1], Step [300/3277], Training Loss: 0.2636, Validation Loss: 0.0333
Epoch [1/1], Step [400/3277], Training Loss: 0.3414, Validation Loss: 0.0314
Process stop at epoch [1/1] with patience 10 within tolerance 0.001



### Test Model with Accuracy and Correct Labels

In [12]:
main.test()

loss of Box on the 104960 test dataset: 0.2556353509426117.


OrderedDict([('problem', 'classification'),
             ('accuracy', '95.99942835365853%'),
             ('num_classes', 29),
             ('column', ('label name', ('true numbers', 'total numbers'))),
             ('labels',
              {'montrealAPT': [100761, 104857],
               'montrealComradeCircle': [100761, 104857],
               'montrealCryptConsole': [100761, 104857],
               'montrealCryptXXX': [100761, 104857],
               'montrealCryptoLocker': [100761, 104857],
               'montrealCryptoTorLocker2015': [100761, 104857],
               'montrealDMALocker': [100761, 104857],
               'montrealDMALockerv3': [100761, 104857],
               'montrealEDA2': [100761, 104857],
               'montrealFlyper': [100761, 104857],
               'montrealGlobe': [100761, 104857],
               'montrealGlobeImposter': [100761, 104857],
               'montrealGlobev3': [100761, 104857],
               'montrealJigSaw': [100761, 104857],
               